# RAG Evaluation with Gemini + TruLens 2.x

This notebook evaluates two RAG configurations using:
- **LLM**: Google Gemini 2.5 Flash
- **Embeddings**: gemini-embedding-001 (3072 dimensions)
- **Evaluation**: TruLens 2.8 (OTEL mode) with LiteLLM/Gemini feedback
- **Metrics**: Answer Relevance, Context Relevance, Groundedness, Answer Correctness

## 1. Environment Setup

In [ ]:
# Fix: Jupyter already runs an asyncio event loop.
# nest_asyncio patches it so TruLens OTEL can call asyncio.run() inside Jupyter.
import nest_asyncio
nest_asyncio.apply()

import os
import io
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from pypdf import PdfReader, PdfWriter

load_dotenv()

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY or GOOGLE_API_KEY == "your_google_api_key_here":
    raise ValueError("Please set GOOGLE_API_KEY in your .env file!")

print(f"nest_asyncio applied.")
print(f"API Key loaded (ends with ...{GOOGLE_API_KEY[-4:]})")

## 2. Prepare PDF Document

> **Why full PDF?** The test questions cover topics throughout IPCC Chapter 3.
> Indexing only the first 30 pages causes retrieval failure (Context Relevance = 0).
> We load the **full document** so the retriever can find relevant chunks.

In [ ]:
# Uncomment to download the full IPCC PDF:
# !curl https://www.ipcc.ch/report/ar6/wg2/downloads/report/IPCC_AR6_WGII_Chapter03.pdf --output IPCC_AR6_WGII_Chapter03.pdf

# Check the full PDF exists
pdf_path = "IPCC_AR6_WGII_Chapter03.pdf"
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"{pdf_path} not found. Please download it first.")

reader = PdfReader(pdf_path)
print(f"Full PDF loaded: {len(reader.pages)} pages total.")
print(f"Using full document for indexing (better retrieval coverage).")

## 3. Initialize Gemini LLM and Embeddings

In [ ]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

llm = GoogleGenAI(
    model="gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
    temperature=0.1
)

embed_model = GoogleGenAIEmbedding(
    model_name="gemini-embedding-001",
    api_key=GOOGLE_API_KEY
)

Settings.llm = llm
Settings.embed_model = embed_model

print("Gemini LLM (gemini-2.5-flash) initialized.")
print("Gemini Embedding (gemini-embedding-001, 3072-dim) initialized.")

## 4. Load Documents (Full PDF)

In [ ]:
from llama_index.core import SimpleDirectoryReader

# Load the FULL PDF — not just first 30 pages
# This ensures the retriever can find answers to all 10 test questions
documents = SimpleDirectoryReader(
    input_files=["IPCC_AR6_WGII_Chapter03.pdf"]
).load_data()

print(f"Loaded {len(documents)} document pages from the full PDF.")
print(f"(Using full document ensures better context coverage for all test questions)")

## 5. Index-Building Functions

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceWindowNodeParser

def build_basic_index(documents):
    """Build a standard vector store index using global Settings."""
    print(f"  Indexing {len(documents)} pages...")
    return VectorStoreIndex.from_documents(documents=documents, show_progress=True)

def build_sentence_window_index(documents):
    """Build a sentence-window index for enhanced context retrieval."""
    node_parser = SentenceWindowNodeParser.from_defaults(
        window_size=3,
        window_metadata_key="window",
        original_text_metadata_key="original-text"
    )
    original_parser = Settings.node_parser
    Settings.node_parser = node_parser
    try:
        print(f"  Indexing {len(documents)} pages with sentence windows...")
        sentence_index = VectorStoreIndex.from_documents(
            documents=documents, show_progress=True
        )
    finally:
        Settings.node_parser = original_parser
    return sentence_index

print("Index builder functions defined.")

## 6. Configure TruLens Evaluation (TruLens 2.8 `Metric` API)

In [ ]:
from trulens.core import TruSession, Metric
from trulens.apps.llamaindex import TruLlama
from trulens.providers.litellm import LiteLLM
from trulens.feedback.groundtruth import GroundTruthAgreement

session = TruSession()
session.reset_database()

os.environ["GEMINI_API_KEY"] = GOOGLE_API_KEY
gemini_provider = LiteLLM(model_engine="gemini/gemini-2.5-flash")

qa_df = pd.read_csv("ipcc_test_questions.csv")
qa_set = [{"query": item["Question"], "response": item["Answer"]} for index, item in qa_df.iterrows()]
print(f"Loaded {len(qa_set)} test Q&A pairs.")
print()
print("Ground truth Q&A pairs:")
for i, qa in enumerate(qa_set):
    print(f"  Q{i+1}: {qa['query'][:60]}...")

# 1. Answer Relevance — does the answer address the question?
m_answer_relevance = (
    Metric(implementation=gemini_provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input_output()
)

# 2. Context Relevance — is retrieved context relevant to the question?
m_context_relevance = (
    Metric(implementation=gemini_provider.context_relevance_with_cot_reasons, name="Context Relevance")
    .on_prompt()
    .on_context(collect_list=True)
)

# 3. Groundedness — is the answer based on the retrieved context?
m_groundedness = (
    Metric(implementation=gemini_provider.groundedness_measure_with_cot_reasons, name="Groundedness")
    .on_context(collect_list=True)
    .on_response()
)

# 4. Answer Correctness — does the answer match ground truth?
m_groundtruth = (
    Metric(
        implementation=GroundTruthAgreement(qa_set, provider=gemini_provider).agreement_measure,
        name="Answer Correctness"
    )
    .on_input_output()
)

metrics = [m_answer_relevance, m_context_relevance, m_groundedness, m_groundtruth]

def get_trulens_recorder(query_engine, app_name):
    return TruLlama(query_engine, app_name=app_name, feedbacks=metrics)

print()
print("TruLens 2.8 configured with 4 metrics:")
for m in metrics:
    print(f"  - {m.name}")

## 7. Run Evals — Basic Index

**Expected good scores with full PDF:** Context Relevance > 0.6, Groundedness > 0.6, Answer Relevance > 0.7

In [ ]:
print("Building Basic Index (indexing full PDF — may take 5-10 minutes)...")
basic_query_index = build_basic_index(documents)
basic_query_engine = basic_query_index.as_query_engine()
basic_recorder = get_trulens_recorder(basic_query_engine, app_name="Basic Query Engine")
print("Basic Index built. TruLens recorder ready.")

In [ ]:
print(f"Running {len(qa_set)} queries through Basic Query Engine...")
with basic_recorder as recording:
    for i, q in enumerate(qa_set):
        print(f"  [{i+1}/{len(qa_set)}] {q['query'][:70]}")
        basic_query_engine.query(q['query'])
print("\nBasic Index evaluation complete.")

## 8. Run Evals — Sentence Window Index

**Sentence Window** should score *higher* than Basic Index because it retrieves surrounding sentences for richer context.

In [ ]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

print("Building Sentence Window Index (may take 5-10 minutes)...")
sentence_window_index = build_sentence_window_index(documents)

postproc = MetadataReplacementPostProcessor(target_metadata_key="window")
sentence_query_engine = sentence_window_index.as_query_engine(
    node_postprocessors=[postproc]
)

sentence_recorder = get_trulens_recorder(
    sentence_query_engine, app_name="Sentence Window Query Engine"
)
print("Sentence Window Index built. TruLens recorder ready.")

In [ ]:
print(f"Running {len(qa_set)} queries through Sentence Window Query Engine...")
with sentence_recorder as recording:
    for i, q in enumerate(qa_set):
        print(f"  [{i+1}/{len(qa_set)}] {q['query'][:70]}")
        sentence_query_engine.query(q['query'])
print("\nSentence Window Index evaluation complete.")

## 9. View Results & Launch Dashboard

### Score Interpretation Guide
| Score | Meaning |
|---|---|
| 0.0 – 0.4 | 🛑 Poor — pipeline broken or data mismatch |
| 0.4 – 0.6 | 🟡 Mediocre — needs improvement |
| 0.6 – 0.8 | 🟢 Good — working reasonably well |
| 0.8 – 1.0 | 🌟 Excellent — RAG pipeline is well-tuned |

### What to compare:
- **Basic vs Sentence Window** — Sentence Window should score higher on Context Relevance and Groundedness
- **Answer Correctness** — how close the LLM answers are to the expected ground truth answers

In [ ]:
records, feedback = session.get_records_and_feedback(app_ids=[])
print(f"Total records collected: {len(records)}")
print()

# Show per-app summary
if len(records) > 0:
    metric_cols = [c for c in records.columns if c in 
                   ["Answer Relevance", "Context Relevance", "Groundedness", "Answer Correctness"]]
    summary = records.groupby("app_name")[metric_cols].mean().round(3)
    print("=== Average Scores per RAG Configuration ===")
    print(summary.to_string())

records.head(10)

In [ ]:
# Launch TruLens Dashboard at http://localhost:8501
# Switch between apps in the dashboard to compare Basic vs Sentence Window
session.run_dashboard()